# Generic FEniCSx to see how stuff works

In [ ]:
import numpy as np
import ufl

from mpi4py import MPI

import dolfinx
import dolfinx.fem.petsc
import dolfinx.mesh
import basix.ufl
from petsc4py import PETSc

length, height = 2., 1.0
Nx, Ny = 2,2
domain = dolfinx.mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0., 0.]), np.array([length, height])],
    [Nx, Ny],
    cell_type=dolfinx.mesh.CellType.quadrilateral,
)

dim = domain.topology.dim
print(f"Mesh topology dimension d={dim}.")

degree = 3
#shape = (dim,)  # this means we want a vector field of size `dim`
v_elem = basix.ufl.element(
    "Lagrange", 
    domain.topology.cell_name(), 
    degree, 
    shape=(dim,)
)
V = dolfinx.fem.functionspace(domain, v_elem)

u_sol = dolfinx.fem.Function(V, name="Displacement")

E = dolfinx.fem.Constant(domain, 10.)
nu = dolfinx.fem.Constant(domain, 0.3)

lmbda = E * nu / (1. + nu) / (1. - 2. * nu)
mu = E / 2. / (1. + nu)

def up_bottom_boundary(x):
    on_bottom = np.isclose(x[1], 0.)
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top
facet_dim = domain.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(domain, facet_dim, up_bottom_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    domain,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)

def epsilon(v):
    return ufl.sym(ufl.grad(v))


def sigma(v):
    return lmbda * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu * epsilon(v)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

rho = 2e-3
g = 9.81
#f = dolfinx.fem.Constant(domain, np.array([0., -rho * g], dtype=dolfinx.default_scalar_type))
x = ufl.SpatialCoordinate(domain)
u_exact_x = 0.1*x[0]*(length-x[0])**2#*(ufl.tanh(height+x[1]))
#u_exact_x = dolfinx.default_scalar_type(0.0)
epsilon_shift = 1e-5
r =1.# ufl.sqrt((x[0]+epsilon_shift)**2 + (x[1]+epsilon_shift)**2)
#u_exact_y = 0.1*(ufl.sin(ufl.pi*x[0]/length)**2) * (height-x[1])
u_exact_y = x[0] * (length - x[0]) * (1+ x[1])

u_exact = ufl.as_vector((u_exact_x, u_exact_y))
f_mms = -ufl.div(sigma(u_exact))
T_mms = sigma(u_exact)*ufl.FacetNormal(domain)


custom_metadata = {"quadrature_degree": 4}
custom_dx = ufl.Measure("dx", domain=domain, metadata=custom_metadata)
custom_ds = ufl.Measure("ds", domain=domain, subdomain_data=boundary_tags, metadata=custom_metadata)
a = ufl.inner(sigma(u), epsilon(v)) * custom_dx
L = ufl.inner(f_mms, v) * custom_dx + ufl.inner(T_mms, v)*custom_ds(1)

def left(x):
    return np.isclose(x[0], 0.)


def right(x):
    return np.isclose(x[0], length)


left_dofs = dolfinx.fem.locate_dofs_geometrical(V, left)
right_dofs = dolfinx.fem.locate_dofs_geometrical(V, right)
zero_vec = np.zeros(dim, dtype=dolfinx.default_scalar_type)
bcs = [
    dolfinx.fem.dirichletbc(zero_vec, left_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, right_dofs, V),
]
problem = dolfinx.fem.petsc.LinearProblem(
    a, L, u=u_sol, bcs=bcs,
    petsc_options_prefix="linear_elasticity",
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps"}
)
problem.solve()
# A = dolfinx.fem.petsc.assemble_matrix(a, bcs=bcs)
# A.assemble()

# # Assemble b
# b = dolfinx.fem.petsc.assemble_vector(L)
# dolfinx.fem.petsc.apply_lifting(b, [a], bcs=[bcs])
# b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES,
#               mode=PETSc.ScatterMode.REVERSE)
# dolfinx.fem.petsc.set_bc(b, bcs)
b_lagrange = problem.b
A_lagrange = problem.A

In [ ]:
A_lagrange.getSize()

In [ ]:
# Assuming u_sol is your numerical solution, and u_exact is your reference
# (u_exact must be a dolfinx.fem.Function or a UFL spatial expression)

# Define the error
e = u_sol - u_exact

# 1. L2 Error
error_L2_form = dolfinx.fem.form(ufl.inner(e, e) * custom_dx)
error_L2 = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_L2_form), op=MPI.SUM))

# 2. H1 Error
error_H1_form = dolfinx.fem.form((ufl.inner(e, e) + ufl.inner(ufl.grad(e), ufl.grad(e))) * custom_dx)
error_H1 = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_H1_form), op=MPI.SUM))

# 3. Energy Error (using your previously defined sigma and epsilon functions)
error_energy_form = dolfinx.fem.form(ufl.inner(sigma(e), epsilon(e)) * custom_dx)
error_energy = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_energy_form), op=MPI.SUM))

print(f"L2 Error:     {error_L2:.2e}")
print(f"H1 Error:     {error_H1:.2e}")
print(f"Energy Error: {error_energy:.2e}")
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")


In [ ]:
import pyvista
from dolfinx.plot import vtk_mesh

topology, cell_types, geometry = vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# The solution array is 1D. We reshape it to N x 2 (since dim=2)
u_values = u_sol.x.array.reshape(-1, dim)

# PyVista requires 3D vectors to warp the mesh. We pad the 2D vectors with Z=0.
u_3d = np.zeros((u_values.shape[0], 3), dtype=np.float64)
u_3d[:, :dim] = u_values

# Attach the 3D displacement vectors to the grid
grid.point_data["Displacement"] = u_3d
grid.set_active_vectors("Displacement")

# Warp the grid by the displacement. 
# We use a factor (e.g., 1000) to exaggerate the deformation so it's visible.
warp_factor = 1.0
warped_grid = grid.warp_by_vector("Displacement", factor=warp_factor)

# Plotting
plotter = pyvista.Plotter()
plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# Show the original undeformed mesh as a wireframe
plotter.add_mesh(grid, style="wireframe", color="black", opacity=0.3, label="Undeformed")

# Show the deformed mesh
plotter.add_mesh(warped_grid, show_edges=False, scalars="Displacement", cmap="coolwarm", label="Deformed")

plotter.view_xy()  # Set camera to view the X-Y plane directly
plotter.show(jupyter_backend="static")

# THB-Splines

In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl

import numpy.typing as npt

from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem_vector_field
from thbsplines.fenicsx.postprocessing import map_spline_to_legendre_vector_field
from thbsplines.fenicsx.kernels import make_vector_bilinear_kernel, make_vector_linear_kernel
from thbsplines.fenicsx.adaptivity import dorfler_marking

p0 = 2
L = 2.
h=1.
n_refinements = 3
knotsx = np.array([0., L/2., L], dtype=np.float64)
knotsx = refine(knotsx, p=p0, n_times=n_refinements)
#log_initial_mesh_size = np.log2(np.max(np.diff(knotsx)))
knotsy = refine(np.array([0., h/2., h], dtype=np.float64), p=p0, n_times=n_refinements)
err_cells = {}
hs = HierarchicalSpace(knots=[knotsx, knotsy], degrees=[p0])
        

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=False, refine_T_neighbours=True, m=3)
hs.hmesh.plot_cells()

In [ ]:
disconnected_mesh, thb_operators, N_max, _ = build_mesh(hs=hs)

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
def up_bottom_boundary(x):
    on_bottom = np.isclose(x[1], knotsy[0])
    on_top = np.isclose(x[1], knotsy[-1])
    return on_bottom|on_top

def left_right_boundaries(x):
    on_left = np.isclose(x[0], knotsx[0])
    on_right = np.isclose(x[0], knotsx[-1])
    return on_left|on_right

dim = disconnected_mesh.topology.dim
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    shape=(dim,),
    lagrange_variant=basix.LagrangeVariant.legendre
)

V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
# print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
facet_dim = disconnected_mesh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, up_bottom_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    disconnected_mesh,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)
custom_metadata = {"quadrature_degree": 8}
ds_custom = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=boundary_tags, metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)
#u_sol = dolfinx.fem.Function(V, name="Displacement")

#E = dolfinx.fem.Constant(disconnected_mesh, 210e3)
#nu = dolfinx.fem.Constant(disconnected_mesh, 0.3)

#lmbda = E * nu / (1. + nu) / (1. - 2. * nu)
E=10.
nu = 0.3
lmbda = E*nu/(1.+nu)/(1.-2.*nu)
#mu = E / 2. / (1. + nu)
mu = E/2./(1.+nu)
lmbda_c = dolfinx.fem.Constant(disconnected_mesh, lmbda)
mu_c = dolfinx.fem.Constant(disconnected_mesh, mu)

def epsilon(v):
    return ufl.sym(ufl.grad(v))

def sigma(v):
    return lmbda_c * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu_c * epsilon(v)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
rho = 2e-3
g = 9.81
my_x = ufl.SpatialCoordinate(disconnected_mesh)

#u_exact_x = dolfinx.default_scalar_type(0.0)
u_exact_x = 0.1*my_x[0] * (L - my_x[0])**2 * (ufl.tanh(h+my_x[1]))
#epsilon_shift = 1e-5
#r =1.# ufl.sqrt((x[0]+epsilon_shift)**2 + (x[1]+epsilon_shift)**2)
u_exact_y = 0.1*(ufl.sin(ufl.pi*my_x[0]/L)**2) * (2.*h-my_x[1])
#u_exact_y = my_x[0] * (L - my_x[0]) * (1.+ my_x[1])
u_exact = ufl.as_vector((u_exact_x, u_exact_y))
f_mms = -ufl.div(sigma(u_exact))
T_mms = sigma(u_exact)* ufl.FacetNormal(disconnected_mesh)

a = ufl.inner(sigma(u), epsilon(v)) * dx_custom
L_cell = ufl.inner(f_mms, v) * dx_custom 
L_facet = ufl.inner(T_mms, v)*ds_custom(1) 


msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcx_L_cell, _, _ = ffcx_jit(msh.comm, L_cell, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernel_L_cell = getattr(ufcx_L_cell.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcx_L_facet, _, _ = ffcx_jit(msh.comm, L_facet, form_compiler_options={"scalar_type": dtype})
kernel_L_facet = getattr(ufcx_L_facet.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, N_max=N_max, morton=False)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh,
                                      N_max=N_max, thb_operators=thb_operators)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh, 
                               N_max=N_max, mult_factor=2)
local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_vector_bilinear_kernel(dtype, rtype, ufcx_kernel=kernela0, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_L_cell = make_vector_linear_kernel(dtype, rtype, ufcx_kernel=kernel_L_cell, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_L_facet = make_vector_linear_kernel(dtype, rtype, ufcx_kernel=kernel_L_facet, padded_dofs=N_max, local_dofs=local_dofs)

In [ ]:
facet_dim = msh.topology.dim-1

boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, up_bottom_boundary)
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for f in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(f)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == f)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
cpp_constants = [lmbda_c._cpp_object, mu_c._cpp_object]
#cpp_constants = []
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=cpp_constants,
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_L_cell.address, cells, np.array([0], dtype=np.int8))],
                 dolfinx.fem.IntegralType.exterior_facet: [(0, tabulate_L_facet.address, boundary_entities, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of stuff to integrate
        coefficients=[C_func._cpp_object], # holds C@T
        constants=cpp_constants, need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
def get_spline_indices_left_right(hs, dofmap):
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        right_dirichlet_indices = np.isclose(basis[:, 1, -hs.degrees[0]-1:], np.full((hs.degrees[0]+1), fill_value=L, dtype=np.float64))
        right_dirichlet_indices = np.all(right_dirichlet_indices, axis=-1)
        right_dirichlet_indices = np.nonzero(right_dirichlet_indices)[0]

        left_dirichlet_indices = np.isclose(basis[:, 1, :-1], np.zeros((hs.degrees[0]+1), dtype=np.float64))
        left_dirichlet_indices = np.all(left_dirichlet_indices, axis=-1)
        left_dirichlet_indices = np.nonzero(left_dirichlet_indices)[0]

        # (A\cap B)\cup(A\cap C) = A\cap(B\cup C)
        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  np.union1d(right_dirichlet_indices,left_dirichlet_indices),
                                                  assume_unique=True)
    pass
    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices
forbidden_indices = get_spline_indices_left_right(hs, dofmap)
if forbidden_indices is not None and len(forbidden_indices) > 0:
    forbidden_indices_vec = np.empty(2 * len(forbidden_indices), dtype=np.int32)
    forbidden_indices_vec[0::2] = 2 * forbidden_indices      # X DOFs
    forbidden_indices_vec[1::2] = 2 * forbidden_indices + 1  # Y DOFs
    forbidden_indices = forbidden_indices_vec

In [ ]:
x_vec = solve_problem_vector_field(hs=hs, a = a_cond, lhs=l_cond, dirichlet_indices=forbidden_indices, 
                                   dummy_index=np.max(padded_cells_to_dofs), V_spline=V_spline)

In [ ]:
# Map the global B-spline coefficients back to local Legendre coefficients
u_dg = map_spline_to_legendre_vector_field(hs, V, C_func, N_max, msh, padded_cells_to_dofs, x_vec)
u_dg.x.scatter_forward()

e = u_dg-u_exact

# Compute L2 Error: sqrt( \int (u_bar - u_dg)^2 dx )
error_L2_form = dolfinx.fem.form(ufl.inner(e,e) * dx_custom)
error_L2_sq = dolfinx.fem.assemble_scalar(error_L2_form)
l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_L2_sq, op=MPI.SUM))

# Compute H1 Semi-norm (Gradient) Error: sqrt( \int |grad(u_bar) - grad(u_dg)|^2 dx )
# This is the "energy" error and is crucial for elliptic PDEs!
error_H1_form = dolfinx.fem.form((ufl.inner(e,e)+ufl.inner(ufl.grad(e), ufl.grad(e))) * dx_custom)
error_H1_sq = dolfinx.fem.assemble_scalar(error_H1_form)
h1_error = np.sqrt(disconnected_mesh.comm.allreduce(error_H1_sq, op=MPI.SUM))

# 3. Energy Error (using your previously defined sigma and epsilon functions)
error_energy_form = dolfinx.fem.form(ufl.inner(sigma(e), epsilon(e)) * dx_custom)
error_energy = np.sqrt(disconnected_mesh.comm.allreduce(dolfinx.fem.assemble_scalar(error_energy_form), op=MPI.SUM))


# Print results
print(f"Absolute L2 Error: {l2_error:.2e}")
#print(f"Relative L2 Error: {l2_error / exact_L2_norm:.2e}\n")

print(f"Absolute H1 Error: {h1_error:.2e}")
#print(f"Relative H1 Error: {h1_error / exact_H1_norm:.2e}")
print(f"Energy Error: {error_energy:.2e}")
print(f"dofs = {x_vec.shape[0]/2}")
#ksp.destroy()
#A.destroy()
#b.destroy()

In [ ]:
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)

#hQ = ufl.CellDiameter(disconnected_mesh)
#volume_form = dolfinx.fem.form(1.0*v*dx_custom)
#cell_volumes = dolfinx.fem.assemble_vector(volume_form).array

# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
e = u_dg-u_exact
local_error_form = dolfinx.fem.form(ufl.inner(sigma(e), epsilon(e)) * v * dx_custom)

err_cells = dorfler_marking(hs, 0.5, local_error_form)

In [ ]:
import pyvista
import dolfinx
from dolfinx.plot import vtk_mesh

dim = msh.geometry.dim

# 1. Create a plot-friendly function space: Continuous Galerkin (Lagrange) degree 1
# This guarantees that DOFs match the physical vertices of the mesh.
V_plot = dolfinx.fem.functionspace(msh, ("Lagrange", 1, (dim,)))

# 2. Interpolate your Legendre/DG function into this nodal space
u_plot = dolfinx.fem.Function(V_plot)
u_plot.interpolate(u_dg)

# 3. Generate the VTK mesh from the plotting space
topology, cell_types, geometry = vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Extract the solution array and reshape it to N x dim
# Because V_plot is a vector space, DOFs are interleaved (x0, y0, x1, y1...)
u_values = u_plot.x.array.reshape(-1, dim)

# 5. PyVista requires 3D vectors to warp the mesh. Pad the 2D vectors with Z=0.
u_3d = np.zeros((u_values.shape[0], 3), dtype=np.float64)
u_3d[:, :dim] = u_values

# 6. Attach the 3D displacement vectors to the grid
grid.point_data["Displacement"] = u_3d
grid.set_active_vectors("Displacement")

# Optional: Calculate displacement magnitude to use for coloring
grid.point_data["Displacement_Magnitude"] = np.linalg.norm(u_3d, axis=1)

# 7. Warp the grid by the displacement
warp_factor = 1.0
warped_grid = grid.warp_by_vector("Displacement", factor=warp_factor)

# 8. Plotting
# plotter = pyvista.Plotter()
# plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# # Show the original undeformed mesh as a wireframe
# plotter.add_mesh(grid, style="wireframe", color="black", opacity=0.3, label="Undeformed")

# # Show the deformed mesh, coloring it by the magnitude of the displacement
# plotter.add_mesh(
#     warped_grid, 
#     show_edges=False, 
#     scalars="Displacement_Magnitude", # Color by magnitude instead of the vector array
#     cmap="coolwarm", 
#     label="Deformed"
# )

# plotter.view_xy()  # Set camera to view the X-Y plane directly
# plotter.show(jupyter_backend="static")